# 15.3 - RAG Evaluation

Status: VERIFIED

## What Are We Solving?
RAG failures come from two sources: bad retrieval (wrong context) and bad generation (ignoring good context). If you only measure the final answer, you cannot tell which part failed.

## Mental Model
RAG evaluation is a two-stage pipeline: Query -> Retrieval -> Generation -> Answer. We must evaluate each stage separately.

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
print("All imports OK")

All imports OK


## Retrieval Metrics

In [2]:
# Retrieval evaluation metrics
def retrieval_metrics(retrieved_docs: list[str], gold_docs: list[str], k: int = 3) -> dict:
    retrieved_at_k = retrieved_docs[:k]
    retrieved_set = set(d.lower() for d in retrieved_at_k)
    gold_set = set(d.lower() for d in gold_docs)
    
    hits = retrieved_set & gold_set
    
    precision_at_k = len(hits) / max(len(retrieved_set), 1)
    recall_at_k = len(hits) / max(len(gold_set), 1)
    hit_rate = 1.0 if hits else 0.0
    
    # MRR (Mean Reciprocal Rank)
    rr = 0.0
    for i, doc in enumerate(retrieved_docs):
        if doc.lower() in gold_set:
            rr = 1.0 / (i + 1)
            break
    
    return {
        f"precision@{k}": round(precision_at_k, 3),
        f"recall@{k}": round(recall_at_k, 3),
        "hit_rate": hit_rate,
        "mrr": round(rr, 3),
    }

# Test with example
retrieved = ["Aspirin causes stomach upset.", "Ibuprofen is a painkiller.", "Aspirin thins blood."]
gold = ["Aspirin causes stomach upset.", "Aspirin increases bleeding risk."]

metrics = retrieval_metrics(retrieved, gold, k=3)
print("Retrieval Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

Retrieval Metrics:
  precision@3: 0.333
  recall@3: 0.5
  hit_rate: 1.0
  mrr: 1.0


## Generation Metrics: Faithfulness

In [3]:
# Faithfulness: does the answer match the retrieved context?
def faithfulness_score(answer: str, context_docs: list[str]) -> float:
    answer_sentences = [s.strip() for s in answer.split('.') if s.strip()]
    context_text = ' '.join(context_docs).lower()
    
    grounded_count = 0
    for sentence in answer_sentences:
        words = sentence.lower().split()
        # Check if key words appear in context
        key_words = [w for w in words if len(w) > 3]
        if key_words:
            matches = sum(1 for w in key_words if w in context_text)
            if matches / len(key_words) > 0.3:
                grounded_count += 1
    
    return grounded_count / max(len(answer_sentences), 1)

answer = "Aspirin can cause stomach upset and increases bleeding risk."
context = ["Aspirin can cause stomach upset, nausea, and increased bleeding risk.", 
           "Aspirin is used to reduce pain and fever."]

score = faithfulness_score(answer, context)
print(f"Faithfulness: {score:.3f}")
print(f"  Answer grounded in context: {score > 0.5}")

Faithfulness: 1.000
  Answer grounded in context: True


## End-to-End RAG Evaluation

In [4]:
# Simple BLEU-like score
def compute_bleu(reference: str, hypothesis: str) -> float:
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    if not hyp_tokens:
        return 0.0
    matches = sum(1 for t in hyp_tokens if t in ref_tokens)
    precision = matches / len(hyp_tokens)
    # Brevity penalty
    bp = min(1.0, len(hyp_tokens) / max(len(ref_tokens), 1))
    return bp * precision

# Complete RAG evaluation pipeline
def evaluate_rag_system(queries: list[dict]) -> dict:
    all_metrics = []
    
    for item in queries:
        q = item["query"]
        retrieved = item["retrieved"]
        gold_docs = item["gold_docs"]
        answer = item["answer"]
        expected = item["expected_answer"]
        
        # Retrieval metrics
        ret_met = retrieval_metrics(retrieved, gold_docs, k=3)
        
        # Generation metrics
        faith = faithfulness_score(answer, retrieved)
        bleu = compute_bleu(expected, answer)
        
        all_metrics.append({
            "query": q,
            "retrieval": ret_met,
            "faithfulness": round(faith, 3),
            "bleu": round(bleu, 3),
        })
    
    # Aggregate
    avg_precision = np.mean([m["retrieval"]["precision@3"] for m in all_metrics])
    avg_recall = np.mean([m["retrieval"]["recall@3"] for m in all_metrics])
    avg_faith = np.mean([m["faithfulness"] for m in all_metrics])
    avg_bleu = np.mean([m["bleu"] for m in all_metrics])
    
    return {
        "num_queries": len(queries),
        "avg_precision@3": round(avg_precision, 3),
        "avg_recall@3": round(avg_recall, 3),
        "avg_faithfulness": round(avg_faith, 3),
        "avg_bleu": round(avg_bleu, 3),
        "per_query": all_metrics,
    }

# Test data
test_queries = [
    {
        "query": "What are side effects of aspirin?",
        "retrieved": ["Aspirin causes stomach upset.", "Aspirin thins blood.", "Ibuprofen is similar."],
        "gold_docs": ["Aspirin causes stomach upset.", "Aspirin increases bleeding risk."],
        "answer": "Aspirin can cause stomach upset and bleeding.",
        "expected_answer": "Side effects include stomach upset and increased bleeding risk."
    },
    {
        "query": "How does paracetamol work?",
        "retrieved": ["Paracetamol reduces pain.", "Aspirin is different.", "Paracetamol acts on the brain."],
        "gold_docs": ["Paracetamol reduces pain by acting on the brain.", "Paracetamol is an analgesic."],
        "answer": "Paracetamol works by reducing pain signals in the brain.",
        "expected_answer": "Paracetamol acts on the brain to reduce pain."
    }
]

results = evaluate_rag_system(test_queries)
print("RAG Evaluation Results:")
print(f"  Queries evaluated: {results['num_queries']}")
print(f"  Avg Precision@3:   {results['avg_precision@3']}")
print(f"  Avg Recall@3:      {results['avg_recall@3']}")
print(f"  Avg Faithfulness:  {results['avg_faithfulness']}")
print(f"  Avg BLEU:          {results['avg_bleu']}")

RAG Evaluation Results:
  Queries evaluated: 2
  Avg Precision@3:   0.166
  Avg Recall@3:      0.25
  Avg Faithfulness:  1.0
  Avg BLEU:          0.278


In [5]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Retrieval quality
ret_metrics = ["avg_precision@3", "avg_recall@3"]
ret_vals = [results[m] for m in ret_metrics]
axes[0].bar(ret_metrics, ret_vals, color=['steelblue', 'coral'])
axes[0].set_title('Retrieval Quality')
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

# Generation quality
gen_metrics = ["avg_faithfulness", "avg_bleu"]
gen_vals = [results[m] for m in gen_metrics]
axes[1].bar(gen_metrics, gen_vals, color=['seagreen', 'goldenrod'])
axes[1].set_title('Generation Quality')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

# Per-query breakdown
query_labels = [f"Q{i+1}" for i in range(len(results["per_query"]))]
precisions = [m["retrieval"]["precision@3"] for m in results["per_query"]]
faiths = [m["faithfulness"] for m in results["per_query"]]
x = np.arange(len(query_labels))
axes[2].bar(x - 0.2, precisions, 0.4, label='Precision@3', color='steelblue')
axes[2].bar(x + 0.2, faiths, 0.4, label='Faithfulness', color='seagreen')
axes[2].set_xticks(x)
axes[2].set_xticklabels(query_labels)
axes[2].set_title('Per-Query Breakdown')
axes[2].legend()
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rag_eval.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualization saved")

Visualization saved


C:\Users\PC\AppData\Local\Temp\ipykernel_4104\427432548.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Verification
assert results["avg_precision@3"] > 0, "Precision must be positive"
assert results["avg_faithfulness"] > 0, "Faithfulness must be positive"
assert len(results["per_query"]) == 2, "Must have per-query results"
print("VERIFICATION PASSED: Phase 15.3 complete")

VERIFICATION PASSED: Phase 15.3 complete
